# 01 — LLM Behavior and Prompt Anatomy

A controlled, credential-free experiment on the parts of an inference request that teams can observe and version.

## Scenario, experimental question, and success criteria

Northstar Support routes messages to `refund`, `shipping`, `account`, or `unknown`. A team reports that “the prompt stopped working” after a release, but the release changed context placement and sampling too.

**Experimental question.** Which observable request-packet change explains the regression: instruction precision, evidence position, distracting context, or sampling configuration?

**Success criteria.** Run the same 20 sliced cases five times per strategy; measure accuracy, correct abstention, unsupported outputs, instability, latency, and estimated packet tokens; change one variable at a time; preserve `unknown` when evidence is missing.

## Learning objectives and safety boundaries

By the end, you can inspect a request packet, construct a controlled behavior experiment, distinguish repeatability from correctness, and route failures to the instruction, context, configuration, evidence, or application boundary.

This notebook uses a transparent deterministic classifier. It is an experimental-design instrument, not a simulation of model internals. It neither authorizes actions nor claims that a real provider will reproduce these effects.

## Environment and reproducibility

The full evaluation runs offline. Install the repository environment from the root README. The final integration cell is opt-in: each learner supplies their own `OPENAI_API_KEY` and sets `PROMPT_COURSE_PROVIDER=openai`. Never paste a secret into this notebook or commit an `.env` file. One live response checks wiring only; it is not an evaluation.

In [ ]:
from dataclasses import asdict
from importlib.util import module_from_spec, spec_from_file_location
from pathlib import Path
import os
import sys

import matplotlib.pyplot as plt
import pandas as pd

sys.path.insert(0, str(Path('src').resolve()))
module_path = Path('curriculum/beginner/01-llm-behavior-and-prompt-anatomy/lab.py')
spec = spec_from_file_location('course01_behavior_lab', module_path)
lab = module_from_spec(spec)
sys.modules[spec.name] = lab
spec.loader.exec_module(lab)
print({'cases': len(lab.load_cases()), 'strategies': list(lab.VARIANTS)})

## Architecture and internal behavior

```text
instruction + user data + selected evidence + configuration
                         ↓
                 candidate generation
                         ↓
        schema + evidence + policy validation
                         ↓
          answer / clarify / escalate
```

A model repeatedly selects tokens conditional on visible request state. The surrounding application owns source selection, validation, authorization, side effects, and observability. Prompt text can influence behavior; it cannot enforce those controls.

## Freeze the evaluation contract

Expected labels are defined before variants are compared. The suite covers normal, ambiguous, boundary, missing-evidence, multilingual, and injection cases. `unknown` is a correct safety outcome, not a failure bucket to tune away.

In [ ]:
cases = lab.load_cases()
case_frame = pd.DataFrame([{
    'id': case.id, 'slice': case.slice, 'expected': case.expected,
    'evidence_available': case.metadata.get('evidence_available', True)
} for case in cases])
display(pd.crosstab(case_frame['slice'], case_frame['expected']))
assert len(cases) == 20
assert {'missing_evidence', 'injection', 'multilingual'} <= set(case_frame['slice'])

## Baseline — vague request, frozen cases

The baseline says only “Help the customer.” Hypothesis: it will abstain consistently, score well on expected-unknown cases, and fail most clear requests. That makes it safe-looking but not useful.

In [ ]:
baseline_rows = lab.run_strategy('vague', repeats=5)
baseline_metrics = lab.metrics(baseline_rows)
pd.Series(baseline_metrics).round(4)

### Inspect baseline failures

Accuracy and abstention must be read together. A classifier that returns `unknown` for everything is perfectly stable and gets every abstention case right, yet fails the product objective. Stability is not correctness.

In [ ]:
pd.DataFrame(lab.slice_accuracy(baseline_rows)).set_index('slice').round(3)

## Step 1 — define a stable packet

The stable packet changes only the instruction: it names the four allowed outcomes and requires approved evidence. Temperature is zero, evidence is first, and no distracting context is included.

In [ ]:
stable_rows = lab.run_strategy('stable', repeats=5)
stable_metrics = lab.metrics(stable_rows)
pd.DataFrame({'vague': baseline_metrics, 'stable': stable_metrics}).T.round(4)
assert stable_metrics['accuracy'] > baseline_metrics['accuracy']
assert stable_metrics['unsupported_rate'] == 0

## Step 2 — change one packet variable at a time

`evidence_middle` changes position, `high_variation` changes temperature, and `overloaded` adds 800 estimated distracting tokens. All three retain the stable instruction and frozen cases. The effects are synthetic and intentionally visible; a real model must be tested rather than assumed to behave identically.

In [ ]:
comparison = pd.DataFrame(lab.compare_strategies(repeats=5)).set_index('strategy')
comparison.round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), constrained_layout=True)
comparison[['accuracy', 'abstention_accuracy', 'instability_rate']].plot.bar(ax=axes[0])
axes[0].set_ylim(0, 1.05); axes[0].set_ylabel('Rate'); axes[0].grid(axis='y', alpha=.25)
comparison['mean_packet_tokens_estimated'].plot.bar(ax=axes[1], color='#C44536')
axes[1].set_ylabel('Estimated tokens'); axes[1].set_title('Request-size tradeoff'); axes[1].grid(axis='y', alpha=.25)
plt.show()

## Step 3 — inspect repeated-run instability

Non-zero temperature does not make answers more or less truthful. Here it creates cross-run label variation. Inspect which case IDs change rather than hiding the effect inside a single average.

In [ ]:
varied = pd.DataFrame([asdict(row) for row in lab.run_strategy('high_variation', repeats=5)])
variation_by_case = varied.groupby(['case_id', 'slice'])['observed'].nunique().rename('distinct_outputs')
variation_by_case[variation_by_case > 1].to_frame()

## Slice analysis

Aggregate accuracy can hide which packet component failed. Compare slices for the stable, position-sensitive, and overloaded variants.

In [ ]:
slice_rows = []
for strategy in ('stable', 'evidence_middle', 'overloaded'):
    for row in lab.slice_accuracy(lab.run_strategy(strategy, repeats=5)):
        slice_rows.append({'strategy': strategy, **row})
pd.DataFrame(slice_rows).pivot(index='slice', columns='strategy', values='accuracy').round(3)

## Failure injection — required evidence is unavailable

A refund-like request arrives while the approved order system is unavailable. The correct result is `unknown`, followed by clarification or human review. Stronger persona wording cannot create evidence.

In [ ]:
missing_case = next(case for case in cases if case.slice == 'missing_evidence')
missing_packet = lab.packet_for(missing_case, 'stable')
missing_outcome = lab.classify(missing_packet)
assert missing_packet.evidence_available is False
assert missing_outcome == 'unknown'
{'case': missing_case.id, 'observed': missing_outcome, 'next_step': 'clarify or escalate'}

## Diagnose the failure

Use the smallest justified repair:

| Symptom | Evidence | Owner | Repair |
| --- | --- | --- | --- |
| vague packet abstains on clear cases | clear-slice failures | instruction contract | specify task and outcomes |
| refund slice regresses only when evidence moves | position-controlled comparison | context assembly | select/reorder sources, then retest |
| outputs differ across repeats | per-case distinct labels | decoding configuration | choose configuration from measured task needs |
| missing source causes refund proposal | unsupported-output rate | evidence and validation | abstain; restore approved source |
| more tokens add no quality | cost/quality comparison | context selection | remove measured waste |

Do not request private chain of thought. Debug with request versions, observable outputs, validation results, slices, and operational telemetry.

## Optional live provider implementation

The adapter returns the same typed `ClassificationResponse` in offline and live modes. To opt in, export your own `OPENAI_API_KEY`, set `PROMPT_COURSE_PROVIDER=openai`, and set `RUN_LIVE=1`. Keep the 20-case evaluation offline unless you have explicitly budgeted a live benchmark.

In [ ]:
if os.getenv('RUN_LIVE') == '1':
    if os.getenv('PROMPT_COURSE_PROVIDER') != 'openai' or not os.getenv('OPENAI_API_KEY'):
        raise RuntimeError('Set your own OPENAI_API_KEY and PROMPT_COURSE_PROVIDER=openai first.')
    provider_result = lab.run_provider_case(cases[0])
    display(provider_result.value.model_dump())
else:
    print('Skipped: offline evaluation is complete; set RUN_LIVE=1 for one explicit integration call.')

## Production upgrade

| Notebook | Production |
| --- | --- |
| transparent classifier | pinned provider adapter behind a stable contract |
| local JSONL | versioned development, held-out, and regression suites |
| estimated packet tokens | provider tokenizer and usage metadata |
| measured local latency | end-to-end p50/p95/p99 by stage and model |
| printed selected fields | privacy-aware traces with contract/model/context versions |
| manual comparison | CI release gate, shadow/canary traffic, alert, rollback |

Pin or record model snapshots and configuration, reserve output capacity, validate outside the model, monitor drift by slice, and never log secrets or raw sensitive content merely to explain a response.

## When not to use an LLM

Prefer deterministic code when policy is explicit, inputs are structured, the decision must be exactly reproducible, or the language flexibility does not justify cost and risk. Do not use prompt text for authentication, tenant isolation, authorization, money movement, or tool permission.

## Review questions and exercises

1. Why can 100% repeatability coexist with poor task accuracy?
2. Which fields must a trace retain to distinguish model drift from context drift?
3. Add a spelling-variant case and state the expected result before running it.
4. Replace the synthetic 800-token distractor with two context blocks and measure which slice changes.
5. Define a release gate that blocks unsupported outputs even if aggregate accuracy improves.

**Advanced challenge.** Repeat the frozen experiment with an approved live model snapshot. Capture provider-reported tokens and latency, estimate confidence intervals across repeated calls, and distinguish sampling variance from model/configuration drift. Do not claim improvement from one output.

## Summary

Prompt quality is measured behavior of a versioned request packet—not a pleasing sentence. Freeze expectations, change one variable at a time, inspect slices and repeated runs, retain safe abstention, and place validation and authorization in application code. Course 02 turns the stable packet into an explicit instruction contract.